[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-9_executed.ipynb)

---



# 100. The Closed-Loop Network Design Problem
## Tier 9: A Hybrid Classical–Quantum Approach with QAOA

### Purpose and scope

This notebook demonstrates a QUBO-based optimization workflow using a state-vector simulation of the Quantum Approximate Optimization Algorithm (QAOA), simulated annealing, and a hybrid strategy that selects the lower-cost result. It focuses on how a binary quadratic objective can be evaluated and explored; it is not a complete operational model of a closed-loop supply chain.

The example uses a randomly generated, symmetric QUBO matrix with 12 binary variables. Those variables are not assigned facility, routing, or product-flow meanings, and the example contains no explicit facility, capacity, demand, or reverse-logistics constraints. Consequently, the notebook illustrates an optimization technique on a toy binary objective rather than solving a fully specified network-design instance.

### Modeling and interpretation notes

- A QUBO objective is unconstrained because every binary vector is allowed. If a practical problem has constraints, they must be represented in the objective, for example by suitable penalty terms; this example does not construct such terms.
- The QAOA state is simulated exactly as a classical complex-valued state vector. This is useful for demonstrating the circuit mathematics at this small size, but it is not execution on quantum hardware and does not demonstrate a quantum speedup or quantum advantage.
- The classical comparison is simulated annealing. Both approaches are stochastic, and their measured runtimes depend on the implementation and machine. Runtime ratios in this notebook are descriptive measurements for this simulation only.
- The five-trial experiment uses independently generated QUBO instances. Its summary is descriptive; it is not a paired statistical test and does not establish statistical significance.

### Workflow

1. Define and evaluate a QUBO objective.
2. Prepare a QAOA state, apply problem and mixer layers, and sample binary assignments.
3. Adjust QAOA parameters with a simple stochastic classical search.
4. Solve the same objective with simulated annealing.
5. Select the lower-cost result and inspect descriptive performance summaries.

The fixed example uses 12 binary variables and 3 QAOA layers. The simulated state vector therefore has \(2^{12}\) amplitudes. State-vector storage and circuit simulation grow exponentially with the number of qubits, so this implementation is intended for small demonstrations.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the stochastic demonstration reproducible within a given environment.
np.random.seed(42)
random.seed(42)

## QUBO model and solution methods

A binary decision vector is \(x \in \{0,1\}^n\). In this notebook, the objective is

$$
f(x) = x^{\mathsf T} Q x + c^{\mathsf T}x,
$$

where \(Q \in \mathbb{R}^{n \times n}\) is the quadratic coefficient matrix and \(c \in \mathbb{R}^n\) contains linear coefficients. The goal is to find a binary vector that minimizes \(f(x)\). With symmetric \(Q\), an off-diagonal interaction contributes through both symmetric entries in \(x^{\mathsf T}Qx\); the code evaluates that expression directly.

### QAOA state and parameter search

QAOA alternates two operations on a quantum state. The problem operation applies a phase determined by the objective, while the mixer operation changes amplitudes between binary assignments. Starting from the uniform superposition gives every assignment an initial amplitude of \(1/\sqrt{2^n}\). Measuring the final state produces candidate binary vectors, which are evaluated using the same QUBO objective.

For layer \(\ell\), the state update is

$$
|\psi_{\ell}\rangle = U_M(\beta_{\ell}) U_C(\gamma_{\ell}) |\psi_{\ell-1}\rangle,
\qquad
U_C(\gamma)|x\rangle = e^{-i\gamma f(x)}|x\rangle,
\qquad
U_M(\beta) = e^{-i\beta\sum_{j=1}^{n} X_j}.
$$

Here \(X_j\) is the Pauli-\(X\) operation on qubit \(j\), and \(\gamma_{\ell}\) and \(\beta_{\ell}\) are classical parameters. The implementation applies the problem phase to each state-vector amplitude and applies the mixer as single-qubit \(X\)-axis rotations. A simple stochastic parameter search proposes small parameter changes and keeps a proposal when its expected objective is no worse. This is a teaching implementation, not a claim that the parameter search is a universal or optimal QAOA optimizer.

The base state is the uniform superposition. The recursion has \(p\) layers, where \(p\) is the QAOA depth. An exact state-vector simulation stores \(2^n\) complex amplitudes; objective values for all basis states also require \(2^n\) entries. These exponential resources limit this direct simulation to small \(n\).

### Simulated annealing and hybrid selection

Simulated annealing begins with a random binary vector, repeatedly flips one bit to form a neighbor, and accepts a lower-cost neighbor. A higher-cost neighbor can also be accepted with probability \(e^{-\Delta/T}\), where \(\Delta\) is the cost increase and \(T\) is the current temperature. Cooling reduces the chance of accepting uphill moves. The hybrid strategy runs QAOA and simulated annealing on the same QUBO instance and returns the solution with the lower objective value. It does not combine their binary vectors.

In [ ]:
@dataclass
class QUBOSolution:
    """Store a binary QUBO solution and descriptive run information."""

    variables: np.ndarray
    objective_value: float
    constraints_satisfied: bool
    solution_quality: float
    computation_time: float


@dataclass
class QuantumCircuit:
    """State-vector QAOA circuit for a small binary optimization problem."""

    num_qubits: int
    num_layers: int
    gamma_params: List[float] = field(default_factory=list)
    beta_params: List[float] = field(default_factory=list)
    circuit_depth: int = 0

    def initialize_parameters(self, num_layers: int) -> None:
        """Draw reproducible initial angles using the notebook's seeded generator."""
        self.num_layers = num_layers
        self.gamma_params = [
            np.random.uniform(0.0, 2.0 * np.pi) for _ in range(num_layers)
        ]
        self.beta_params = [
            np.random.uniform(0.0, np.pi) for _ in range(num_layers)
        ]
        self.circuit_depth = 2 * num_layers

    def apply_mixer_layer(self, state: np.ndarray, beta: float) -> np.ndarray:
        """Apply exp(-i beta X) to each qubit axis of the state vector."""
        amplitudes = state.reshape((2,) * self.num_qubits)
        cosine = np.cos(beta)
        sine = -1j * np.sin(beta)

        for qubit in range(self.num_qubits):
            moved = np.moveaxis(amplitudes, qubit, 0)
            zero_amplitudes = moved[0].copy()
            one_amplitudes = moved[1].copy()
            moved[0] = cosine * zero_amplitudes + sine * one_amplitudes
            moved[1] = sine * zero_amplitudes + cosine * one_amplitudes
            amplitudes = np.moveaxis(moved, 0, qubit)

        return amplitudes.reshape(-1)

    def apply_problem_layer(
        self,
        state: np.ndarray,
        gamma: float,
        objective_values: np.ndarray,
    ) -> np.ndarray:
        """Encode each basis state's QUBO cost as a complex phase."""
        return state * np.exp(-1j * gamma * objective_values)

    def execute_circuit(self, objective_values: np.ndarray) -> np.ndarray:
        """Return normalized measurement probabilities for all basis states."""
        expected_size = 2**self.num_qubits
        if objective_values.shape != (expected_size,):
            raise ValueError("Objective values must contain one entry per basis state.")

        state = np.ones(expected_size, dtype=complex) / np.sqrt(expected_size)
        for layer in range(self.num_layers):
            state = self.apply_problem_layer(
                state, self.gamma_params[layer], objective_values
            )
            state = self.apply_mixer_layer(state, self.beta_params[layer])

        probabilities = np.abs(state) ** 2
        probability_sum = probabilities.sum()
        if not np.isfinite(probability_sum) or probability_sum <= 0.0:
            raise FloatingPointError("The circuit produced invalid measurement probabilities.")

        # Normalization protects sampling from small floating-point accumulation error.
        return probabilities / probability_sum


@dataclass
class QAOA:
    """QAOA with exact state-vector simulation and stochastic angle proposals."""

    qaoa_id: str
    num_qubits: int
    num_layers: int
    max_iterations: int = 100
    convergence_threshold: float = 1e-6
    optimization_history: List[Dict[str, Any]] = field(default_factory=list)

    @staticmethod
    def _evaluate_qubo(
        solution: np.ndarray,
        qubo_matrix: np.ndarray,
        linear_terms: np.ndarray,
    ) -> float:
        """Evaluate x^T Q x + c^T x for one binary vector."""
        return float(solution @ qubo_matrix @ solution + linear_terms @ solution)

    def _all_objective_values(
        self,
        qubo_matrix: np.ndarray,
        linear_terms: np.ndarray,
    ) -> np.ndarray:
        """Evaluate the QUBO objective for every state-vector basis assignment."""
        indices = np.arange(2**self.num_qubits)
        bit_positions = np.arange(self.num_qubits - 1, -1, -1)
        assignments = ((indices[:, None] >> bit_positions) & 1).astype(int)
        return np.einsum(
            "bi,ij,bj->b", assignments, qubo_matrix, assignments
        ) + assignments @ linear_terms

    def optimize_qubo(
        self,
        qubo_matrix: np.ndarray,
        linear_terms: np.ndarray,
    ) -> QUBOSolution:
        """Search QAOA samples while adjusting angles by stochastic proposals."""
        start_time = time.perf_counter()
        self.optimization_history = []
        objective_values = self._all_objective_values(qubo_matrix, linear_terms)

        circuit = QuantumCircuit(self.num_qubits, self.num_layers)
        circuit.initialize_parameters(self.num_layers)
        best_solution = None
        best_value = float("inf")
        previous_best_expected = float("inf")

        for iteration in range(self.max_iterations):
            probabilities = circuit.execute_circuit(objective_values)
            expected_value = float(probabilities @ objective_values)

            # Sampling turns the simulated measurement distribution into candidate decisions.
            sampled_indices = np.random.choice(
                len(probabilities), size=100, p=probabilities
            )
            for index in sampled_indices:
                bits = np.array(
                    [int(bit) for bit in format(int(index), f"0{self.num_qubits}b")]
                )
                value = self._evaluate_qubo(bits, qubo_matrix, linear_terms)
                if value < best_value:
                    best_value = value
                    best_solution = bits.copy()

            self.optimization_history.append(
                {
                    "iteration": iteration + 1,
                    "best_value": best_value,
                    "expected_value": expected_value,
                    "num_samples": len(sampled_indices),
                    "gamma_params": circuit.gamma_params.copy(),
                    "beta_params": circuit.beta_params.copy(),
                }
            )

            if (
                iteration > 0
                and abs(previous_best_expected - expected_value)
                < self.convergence_threshold
            ):
                break
            previous_best_expected = expected_value

            proposed_gamma = [
                (angle + np.random.uniform(-0.1, 0.1)) % (2.0 * np.pi)
                for angle in circuit.gamma_params
            ]
            proposed_beta = [
                (angle + np.random.uniform(-0.1, 0.1)) % np.pi
                for angle in circuit.beta_params
            ]
            old_gamma = circuit.gamma_params
            old_beta = circuit.beta_params
            circuit.gamma_params = proposed_gamma
            circuit.beta_params = proposed_beta
            proposed_probabilities = circuit.execute_circuit(objective_values)
            proposed_expected = float(proposed_probabilities @ objective_values)

            # Keep a parameter proposal only when it does not worsen the expected cost.
            if proposed_expected > expected_value:
                circuit.gamma_params = old_gamma
                circuit.beta_params = old_beta

        if best_solution is None:
            raise RuntimeError("QAOA did not produce a sampled binary assignment.")

        computation_time = time.perf_counter() - start_time
        return QUBOSolution(
            variables=best_solution,
            objective_value=best_value,
            constraints_satisfied=True,
            solution_quality=-best_value,
            computation_time=computation_time,
        )


@dataclass
class ClassicalOptimizer:
    """Simulated annealing optimizer for the same QUBO objective."""

    optimizer_id: str
    optimization_method: str = "simulated_annealing"
    max_iterations: int = 1000
    temperature: float = 100.0
    cooling_rate: float = 0.95

    @staticmethod
    def _evaluate_qubo(
        solution: np.ndarray,
        qubo_matrix: np.ndarray,
        linear_terms: np.ndarray,
    ) -> float:
        """Evaluate x^T Q x + c^T x for one binary vector."""
        return float(solution @ qubo_matrix @ solution + linear_terms @ solution)

    def optimize_qubo(
        self,
        qubo_matrix: np.ndarray,
        linear_terms: np.ndarray,
    ) -> QUBOSolution:
        """Minimize the QUBO objective by accepting or rejecting bit flips."""
        start_time = time.perf_counter()
        num_variables = qubo_matrix.shape[0]
        current_solution = np.random.randint(0, 2, num_variables)
        current_value = self._evaluate_qubo(
            current_solution, qubo_matrix, linear_terms
        )
        best_solution = current_solution.copy()
        best_value = current_value
        temperature = self.temperature

        for _ in range(self.max_iterations):
            neighbor_solution = current_solution.copy()
            flip_index = np.random.randint(0, num_variables)
            neighbor_solution[flip_index] = 1 - neighbor_solution[flip_index]
            neighbor_value = self._evaluate_qubo(
                neighbor_solution, qubo_matrix, linear_terms
            )
            delta = neighbor_value - current_value

            accept_neighbor = delta < 0.0
            if not accept_neighbor and temperature > 0.0:
                accept_neighbor = np.random.random() < np.exp(-delta / temperature)

            if accept_neighbor:
                current_solution = neighbor_solution
                current_value = neighbor_value
                if current_value < best_value:
                    best_solution = current_solution.copy()
                    best_value = current_value

            temperature *= self.cooling_rate

        computation_time = time.perf_counter() - start_time
        return QUBOSolution(
            variables=best_solution,
            objective_value=best_value,
            constraints_satisfied=True,
            solution_quality=-best_value,
            computation_time=computation_time,
        )


@dataclass
class QuantumHybridOptimizer:
    """Run QAOA and simulated annealing, then retain the lower-cost solution."""

    optimizer_id: str
    qaoa: QAOA
    classical_optimizer: ClassicalOptimizer
    hybrid_strategy: str = "quantum_first"
    performance_history: List[Dict[str, Any]] = field(default_factory=list)

    def optimize_network_design(
        self,
        network_matrix: np.ndarray,
        demand_vector: np.ndarray,
    ) -> Dict[str, Any]:
        """Compare both optimizers on one QUBO instance and select the best cost."""
        if network_matrix.shape != (self.qaoa.num_qubits, self.qaoa.num_qubits):
            raise ValueError("The QUBO matrix shape must match the number of variables.")
        if demand_vector.shape != (self.qaoa.num_qubits,):
            raise ValueError("The linear coefficient vector has an incompatible shape.")
        if not np.allclose(network_matrix, network_matrix.T):
            raise ValueError("The QUBO matrix must be symmetric in this example.")
        if self.hybrid_strategy not in {
            "quantum_first", "classical_first", "alternating"
        }:
            raise ValueError("Unknown hybrid strategy.")

        print(f"Running hybrid optimization: {self.hybrid_strategy}")
        results = {
            "quantum_solution": None,
            "classical_solution": None,
            "hybrid_solution": None,
            "performance_comparison": {},
            "speedup_analysis": {},
        }

        if self.hybrid_strategy == "classical_first":
            classical_solution = self.classical_optimizer.optimize_qubo(
                network_matrix, demand_vector
            )
            quantum_solution = self.qaoa.optimize_qubo(
                network_matrix, demand_vector
            )
        else:
            quantum_solution = self.qaoa.optimize_qubo(
                network_matrix, demand_vector
            )
            classical_solution = self.classical_optimizer.optimize_qubo(
                network_matrix, demand_vector
            )

        selected_solution = min(
            (quantum_solution, classical_solution),
            key=lambda solution: solution.objective_value,
        )
        hybrid_solution = QUBOSolution(
            variables=selected_solution.variables.copy(),
            objective_value=selected_solution.objective_value,
            constraints_satisfied=selected_solution.constraints_satisfied,
            solution_quality=selected_solution.solution_quality,
            computation_time=(
                quantum_solution.computation_time
                + classical_solution.computation_time
            ),
        )

        results["quantum_solution"] = quantum_solution
        results["classical_solution"] = classical_solution
        results["hybrid_solution"] = hybrid_solution
        results["performance_comparison"] = {
            "quantum_quality": quantum_solution.solution_quality,
            "classical_quality": classical_solution.solution_quality,
            "hybrid_quality": hybrid_solution.solution_quality,
            "quantum_time": quantum_solution.computation_time,
            "classical_time": classical_solution.computation_time,
            "hybrid_time": hybrid_solution.computation_time,
        }

        quantum_speedup = (
            classical_solution.computation_time / quantum_solution.computation_time
            if quantum_solution.computation_time > 0.0
            else float("nan")
        )
        hybrid_speedup = (
            classical_solution.computation_time / hybrid_solution.computation_time
            if hybrid_solution.computation_time > 0.0
            else float("nan")
        )
        results["speedup_analysis"] = {
            "quantum_speedup": quantum_speedup,
            "hybrid_speedup": hybrid_speedup,
            "objective_improvement": (
                classical_solution.objective_value
                - hybrid_solution.objective_value
            ),
        }

        self.performance_history.append(
            {
                "hybrid_strategy": self.hybrid_strategy,
                "quantum_quality": quantum_solution.solution_quality,
                "classical_quality": classical_solution.solution_quality,
                "hybrid_quality": hybrid_solution.solution_quality,
                "quantum_speedup": quantum_speedup,
                "hybrid_speedup": hybrid_speedup,
            }
        )
        return results

    def generate_quantum_report(self) -> Dict[str, Any]:
        """Summarize observed runs without making hardware or scalability claims."""
        report = {
            "optimizer_id": self.optimizer_id,
            "system_overview": {
                "qaoa_qubits": self.qaoa.num_qubits,
                "qaoa_layers": self.qaoa.num_layers,
                "classical_method": self.classical_optimizer.optimization_method,
                "hybrid_strategy": self.hybrid_strategy,
            },
            "performance_analysis": {},
            "interpretation": (
                "These are descriptive measurements from a classical state-vector "
                "simulation and simulated annealing; they do not establish quantum advantage."
            ),
        }
        if self.performance_history:
            quantum_values = [
                entry["quantum_quality"] for entry in self.performance_history
            ]
            classical_values = [
                entry["classical_quality"] for entry in self.performance_history
            ]
            hybrid_values = [
                entry["hybrid_quality"] for entry in self.performance_history
            ]
            report["performance_analysis"] = {
                "mean_quantum_quality": float(np.mean(quantum_values)),
                "mean_classical_quality": float(np.mean(classical_values)),
                "mean_hybrid_quality": float(np.mean(hybrid_values)),
                "quantum_lower_cost_fraction": float(
                    np.mean(np.array(quantum_values) > np.array(classical_values))
                ),
                "hybrid_lower_cost_fraction": float(
                    np.mean(np.array(hybrid_values) > np.array(classical_values))
                ),
            }
        return report

## Initialize the optimizers and generate the example QUBO

The next cell sets the example size and algorithm parameters, then creates a symmetric quadratic matrix and a linear coefficient vector. These random coefficients define a reproducible mathematical test instance; they do not encode a real facility-location or product-routing dataset. The diagonal entries are replaced with positive values, while off-diagonal interactions may be positive or negative.

In [ ]:
print("Initializing the hybrid optimization system...")
print("=" * 48)

qaoa = QAOA(
    qaoa_id="qaoa_optimizer_001",
    num_qubits=12,
    num_layers=3,
    max_iterations=50,
)

classical_optimizer = ClassicalOptimizer(
    optimizer_id="classical_optimizer_001",
    optimization_method="simulated_annealing",
    max_iterations=1000,
    temperature=100.0,
    cooling_rate=0.95,
)

hybrid_optimizer = QuantumHybridOptimizer(
    optimizer_id="hybrid_optimizer_001",
    qaoa=qaoa,
    classical_optimizer=classical_optimizer,
    hybrid_strategy="quantum_first",
)

print(f"QAOA optimizer: {qaoa.qaoa_id}")
print(f"  Number of qubits: {qaoa.num_qubits}")
print(f"  Number of layers: {qaoa.num_layers}")
print(f"  Maximum parameter-search iterations: {qaoa.max_iterations}")
print(f"\nClassical optimizer: {classical_optimizer.optimizer_id}")
print(f"  Method: {classical_optimizer.optimization_method}")
print(f"  Maximum iterations: {classical_optimizer.max_iterations}")
print(f"  Initial temperature: {classical_optimizer.temperature}")
print(f"\nHybrid optimizer: {hybrid_optimizer.optimizer_id}")
print(f"  Run order: {hybrid_optimizer.hybrid_strategy}")

problem_size = qaoa.num_qubits
qubo_matrix = np.random.uniform(-10.0, 10.0, (problem_size, problem_size))
qubo_matrix = (qubo_matrix + qubo_matrix.T) / 2.0
np.fill_diagonal(
    qubo_matrix, np.random.uniform(0.0, 20.0, problem_size)
)
linear_terms = np.random.uniform(-5.0, 15.0, problem_size)

assert qubo_matrix.shape == (problem_size, problem_size)
assert linear_terms.shape == (problem_size,)
assert np.allclose(qubo_matrix, qubo_matrix.T)

print("\nGenerated QUBO instance:")
print(f"  Number of binary variables: {problem_size}")
print(f"  QUBO matrix shape: {qubo_matrix.shape}")
print(f"  Linear coefficient vector shape: {linear_terms.shape}")
print(f"  Fraction of nonzero matrix entries: {np.count_nonzero(qubo_matrix) / qubo_matrix.size:.2%}")
print(f"  Mean matrix coefficient: {np.mean(qubo_matrix):.2f}")
print(f"  Mean linear coefficient: {np.mean(linear_terms):.2f}")
print(f"  State-vector basis assignments: {2 ** problem_size}")

## Run the hybrid optimization and descriptive trials

For each instance, both optimizers minimize the same objective \(f(x)\). The returned hybrid solution is whichever of their two returned assignments has the lower objective value. The field named `solution_quality` is defined as \(-f(x)\), so larger values correspond exactly to lower objective costs; it is not a normalized percentage or a probability.

The five additional trials use newly generated QUBO instances. Their quality and timing summaries describe these runs only. Because the instances differ, comparing their averages does not isolate an algorithm effect and should not be interpreted as a statistical significance test.

In [ ]:
def run_quantum_optimization(
    hybrid_optimizer: QuantumHybridOptimizer,
    qubo_matrix: np.ndarray,
    linear_terms: np.ndarray,
) -> Dict[str, Any]:
    """Run both optimizers, report their costs, and return the result dictionary."""
    print("Running hybrid optimization...")
    results = hybrid_optimizer.optimize_network_design(qubo_matrix, linear_terms)

    for label, key in (
        ("QAOA state-vector simulation", "quantum_solution"),
        ("Simulated annealing", "classical_solution"),
        ("Selected hybrid result", "hybrid_solution"),
    ):
        solution = results[key]
        recomputed_value = hybrid_optimizer.qaoa._evaluate_qubo(
            solution.variables, qubo_matrix, linear_terms
        )
        assert np.isclose(recomputed_value, solution.objective_value)
        assert np.all((solution.variables == 0) | (solution.variables == 1))
        print(f"\n{label}:")
        print(f"  Objective value (lower is better): {solution.objective_value:.4f}")
        print(f"  Solution quality (-objective): {solution.solution_quality:.4f}")
        print(f"  Runtime: {solution.computation_time:.6f} seconds")
        print(f"  Binary variables: {solution.variables}")

    speedup = results["speedup_analysis"]
    print("\nDescriptive runtime ratios:")
    print(f"  Simulated-annealing time / QAOA simulation time: {speedup['quantum_speedup']:.4f}x")
    print(f"  Simulated-annealing time / total hybrid time: {speedup['hybrid_speedup']:.4f}x")
    print(f"  Objective reduction relative to simulated annealing: {speedup['objective_improvement']:.4f}")
    print("  These ratios measure classical implementations and do not indicate quantum speedup.")
    return results


results = run_quantum_optimization(hybrid_optimizer, qubo_matrix, linear_terms)

num_trials = 5
trial_results = []
print(f"\nRunning {num_trials} descriptive trials on independently generated instances...")

for trial in range(num_trials):
    trial_qubo = np.random.uniform(-10.0, 10.0, (problem_size, problem_size))
    trial_qubo = (trial_qubo + trial_qubo.T) / 2.0
    np.fill_diagonal(
        trial_qubo, np.random.uniform(0.0, 20.0, problem_size)
    )
    trial_linear = np.random.uniform(-5.0, 15.0, problem_size)

    trial_result = hybrid_optimizer.optimize_network_design(
        trial_qubo, trial_linear
    )
    trial_results.append(trial_result)
    print(
        f"  Trial {trial + 1}: QAOA cost = "
        f"{trial_result['quantum_solution'].objective_value:.4f}, "
        f"simulated-annealing cost = "
        f"{trial_result['classical_solution'].objective_value:.4f}, "
        f"selected cost = "
        f"{trial_result['hybrid_solution'].objective_value:.4f}"
    )

quantum_costs = np.array(
    [trial["quantum_solution"].objective_value for trial in trial_results]
)
classical_costs = np.array(
    [trial["classical_solution"].objective_value for trial in trial_results]
)
hybrid_costs = np.array(
    [trial["hybrid_solution"].objective_value for trial in trial_results]
)
quantum_times = np.array(
    [trial["quantum_solution"].computation_time for trial in trial_results]
)
classical_times = np.array(
    [trial["classical_solution"].computation_time for trial in trial_results]
)

print("\nDescriptive trial summary (mean ± standard deviation):")
print(f"  QAOA simulation objective: {quantum_costs.mean():.4f} ± {quantum_costs.std():.4f}")
print(f"  Simulated-annealing objective: {classical_costs.mean():.4f} ± {classical_costs.std():.4f}")
print(f"  Selected hybrid objective: {hybrid_costs.mean():.4f} ± {hybrid_costs.std():.4f}")
print(f"  QAOA simulation runtime: {quantum_times.mean():.6f} ± {quantum_times.std():.6f} seconds")
print(f"  Simulated-annealing runtime: {classical_times.mean():.6f} ± {classical_times.std():.6f} seconds")
print(
    "  QAOA simulation returned a lower cost on "
    f"{np.count_nonzero(quantum_costs < classical_costs)}/{num_trials} instances."
)
print("  This small set of independently generated instances is descriptive, not an inferential test.")

final_report = hybrid_optimizer.generate_quantum_report()
print("\nRun summary:")
print(final_report["system_overview"])
print(final_report["performance_analysis"])
print(final_report["interpretation"])

## Visualize convergence and trial outcomes

The first panel shows the best sampled QUBO cost recorded during the most recent QAOA run. The other panels compare objective values and measured runtimes. Lower objective values are preferable; runtime bars refer to classical execution of the algorithms in this notebook. These plots summarize the observed computation and do not imply quantum hardware performance or general scalability.

In [ ]:
def visualize_quantum_performance(
    hybrid_optimizer: QuantumHybridOptimizer,
    results: Dict[str, Any],
    trial_results: List[Dict[str, Any]],
) -> None:
    """Plot sampled costs, solution costs, runtime, and independent trial costs."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ax1, ax2, ax3, ax4 = axes.flat

    history = hybrid_optimizer.qaoa.optimization_history
    if history:
        iterations = [entry["iteration"] for entry in history]
        sampled_best = [entry["best_value"] for entry in history]
        expected_values = [entry["expected_value"] for entry in history]
        ax1.plot(iterations, sampled_best, marker="o", label="Best sampled objective")
        ax1.plot(iterations, expected_values, linestyle="--", label="Expected objective")
        ax1.set_xlabel("Parameter-search iteration")
        ax1.set_ylabel("Objective value (lower is better)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    ax1.set_title("QAOA state-vector simulation")

    method_labels = ["QAOA simulation", "Simulated annealing", "Hybrid selection"]
    method_costs = [
        results["quantum_solution"].objective_value,
        results["classical_solution"].objective_value,
        results["hybrid_solution"].objective_value,
    ]
    ax2.bar(method_labels, method_costs, color=["steelblue", "seagreen", "indianred"])
    ax2.set_ylabel("Objective value (lower is better)")
    ax2.set_title("Objective comparison on the main instance")
    ax2.tick_params(axis="x", rotation=15)
    ax2.grid(True, axis="y", alpha=0.3)

    method_times = [
        results["quantum_solution"].computation_time,
        results["classical_solution"].computation_time,
        results["hybrid_solution"].computation_time,
    ]
    ax3.bar(method_labels, method_times, color=["steelblue", "seagreen", "indianred"])
    ax3.set_ylabel("Measured runtime (seconds)")
    ax3.set_title("Runtime on the main instance")
    ax3.tick_params(axis="x", rotation=15)
    ax3.grid(True, axis="y", alpha=0.3)

    trial_numbers = np.arange(1, len(trial_results) + 1)
    trial_quantum_costs = [
        trial["quantum_solution"].objective_value for trial in trial_results
    ]
    trial_classical_costs = [
        trial["classical_solution"].objective_value for trial in trial_results
    ]
    trial_hybrid_costs = [
        trial["hybrid_solution"].objective_value for trial in trial_results
    ]
    ax4.plot(trial_numbers, trial_quantum_costs, marker="o", label="QAOA simulation")
    ax4.plot(trial_numbers, trial_classical_costs, marker="s", label="Simulated annealing")
    ax4.plot(trial_numbers, trial_hybrid_costs, marker="^", label="Hybrid selection")
    ax4.set_xlabel("Independent instance number")
    ax4.set_ylabel("Objective value (lower is better)")
    ax4.set_title("Objective values across generated instances")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.suptitle("QUBO optimization: classical simulation results", fontsize=15)
    fig.tight_layout()
    plt.show()

    quantum_solution = results["quantum_solution"]
    classical_solution = results["classical_solution"]
    hybrid_solution = results["hybrid_solution"]
    print("\nInterpretation:")
    print(f"  QAOA simulation objective: {quantum_solution.objective_value:.4f}")
    print(f"  Simulated-annealing objective: {classical_solution.objective_value:.4f}")
    print(f"  Selected hybrid objective: {hybrid_solution.objective_value:.4f}")
    print(
        "  Active binary variables in the selected solution: "
        f"{int(np.sum(hybrid_solution.variables))} of {len(hybrid_solution.variables)}"
    )
    print(
        "  The selected solution has the lower objective of the two returned candidates; "
        "this does not establish global optimality."
    )
    print(
        "  Exact state-vector simulation is classical computation and does not demonstrate "
        "quantum advantage."
    )


visualize_quantum_performance(hybrid_optimizer, results, trial_results)

## Strengths, limitations, and interpretation

### What this demonstration establishes

- The QUBO objective can be evaluated consistently for binary vectors and for every basis assignment used by the state-vector simulator.
- QAOA's problem and mixer operations can be represented explicitly for a small number of binary variables.
- Simulated annealing and QAOA sampling can produce candidate solutions that are compared using the same objective.
- A hybrid selection rule can return the lower-cost candidate from the two methods on a given instance.

### What it does not establish

- The random coefficients do not constitute a closed-loop logistics network model. There are no explicit facility, material-flow, demand, capacity, or service constraints.
- The state-vector calculation runs on classical hardware, whose storage and computation requirements grow exponentially with the number of qubits. It does not measure quantum hardware performance.
- A lower measured runtime in one run would not demonstrate a quantum speedup. The timing comparison is implementation-specific, stochastic, and limited to this small example.
- The returned candidate is the best sampled or annealing solution encountered, not a certificate of global optimality. Multiple binary assignments can also have the same optimal objective value.

### Natural extensions within the same modeling framework

A more operational study could define binary decisions for facility opening and routing, derive the corresponding cost terms, and encode required constraints with carefully scaled penalties. Such a model would need validation that penalty coefficients preserve the intended priorities and that every feasible network decision has a meaningful binary representation. Larger instances would also require attention to the exponential cost of exact state-vector simulation. These extensions are not implemented here.